In [1]:
import pandas as pd

data = pd.read_csv("dataset.csv")
print(data)    

      text_id annotators_id  \
0       1-Jan   ['7', '15']   
1      10-Jan   ['7', '15']   
2       1-100   ['7', '15']   
3      1-1000   ['7', '15']   
4      1-1001  ['10', '18']   
...       ...           ...   
28443   4-995        ['19']   
28444   4-996        ['19']   
28445   4-997        ['19']   
28446   4-998        ['19']   
28447   4-999        ['19']   

                                                    text  \
0      Wajah Jurnalis Dibalut Perban Saat Melaporkan ...   
1      Elektabilitas Paslon 02 sebentar lagi terjun b...   
2      Gini aja deh @KPU_ID usul aja nih. Gimana kalo...   
3      Relawan Tionghoa Kalbar yg di Jakarta mendekla...   
4      Kristen, Hindu, Islam dapat perlakuan istimewa...   
...                                                  ...   
28443          Akun WhatsApp Pj Bupati Tegal Agustyarsah   
28444  Buldoser Israel Menabrak Pengungsi Pasien Ruma...   
28445  @TSolihien Sangat perlu di cek dan ricek kemba...   
28446  Perayaan Chinese New

In [2]:
# Daftar kolom yang wajib ada sesuai Bab 3.1.2 proposal kamu
kolom_dipake = [
    'text',
    'toxicity',
    'polarized',
    'annotators_id',
    'is_noise_or_spam_text'
]

# Memfilter dataset
data = data[kolom_dipake]

# Menampilkan 5 data teratas untuk memastikan kolom sudah lengkap
print(data.head())

                                                text    toxicity   polarized  \
0  Wajah Jurnalis Dibalut Perban Saat Melaporkan ...  ['0', '0']  ['0', '0']   
1  Elektabilitas Paslon 02 sebentar lagi terjun b...  ['0', '0']  ['1', '1']   
2  Gini aja deh @KPU_ID usul aja nih. Gimana kalo...  ['1', '0']  ['0', '0']   
3  Relawan Tionghoa Kalbar yg di Jakarta mendekla...  ['0', '0']  ['1', '0']   
4  Kristen, Hindu, Islam dapat perlakuan istimewa...  ['0', '0']  ['1', '1']   

  annotators_id is_noise_or_spam_text  
0   ['7', '15']            ['0', '0']  
1   ['7', '15']            ['0', '0']  
2   ['7', '15']            ['0', '0']  
3   ['7', '15']            ['0', '0']  
4  ['10', '18']            ['0', '0']  


In [3]:
#PREPROCESSING DATA

#CLEANING DATA

#hapus spam
import ast
# 1. Konversi string "[0, 1]" menjadi list asli [0, 1]
# Gunakan ast.literal_eval agar Python paham itu adalah list, bukan teks biasa
def safe_to_list(val):
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except:
            return val
    return val

# Terapkan konversi pada kolom yang bermasalah
data['is_noise_or_spam_text'] = data['is_noise_or_spam_text'].apply(safe_to_list)

# 2. Fungsi Filter: Cek jika ada angka 1 (spam)
def check_spam(labels):
    if isinstance(labels, list):
        # Kita ubah semua ke string dulu untuk jaga-jaga jika ada campuran tipe data
        str_labels = [str(i) for i in labels]
        return '1' in str_labels
    return False

# 3. Proses Pembersihan Data
jumlah_awal = len(data)

# Hapus baris di mana check_spam mengembalikan True
# Simbol ~ artinya "yang tidak memenuhi kondisi/kebalikan"
data = data[~data['is_noise_or_spam_text'].apply(check_spam)].copy()

print(f"Pembersihan Spam Selesai!")
print(f"Jumlah baris awal: {jumlah_awal}")
print(f"Sisa data sekarang: {len(data)} baris")


Pembersihan Spam Selesai!
Jumlah baris awal: 28448
Sisa data sekarang: 25564 baris


In [4]:

#hapus perfect disagreement
import ast

# 1. Fungsi untuk mengubah string "[ '0', '1']" menjadi list asli [0, 1]
def clean_label_format(val):
    if isinstance(val, str):
        try:
            # ast.literal_eval akan mengubah string "[ '0', '1']" menjadi list ['0', '1']
            res = ast.literal_eval(val)
            # Ubah tiap elemen menjadi integer: [0, 1]
            return [int(i) for i in res]
        except:
            return val
    return val

# Terapkan pembersihan format pada kolom toxicity dan polarized
data['toxicity'] = data['toxicity'].apply(clean_label_format)
data['polarized'] = data['polarized'].apply(clean_label_format)

# 2. Fungsi untuk mengecek rasio 50% (Perfect Disagreement)
def is_50_percent(labels):
    if isinstance(labels, list) and len(labels) > 0:
        # Menghitung rasio angka 1
        ratio = sum(labels) / len(labels)
        return ratio == 0.5
    return False

# 3. Eksekusi Penghapusan sesuai instruksimu
jumlah_awal = len(data)

# Hapus jika salah satu kolom (toxicity ATAU polarized) memiliki rasio 50:50
data = data[~(data['toxicity'].apply(is_50_percent) | 
              data['polarized'].apply(is_50_percent))].copy()

print(f"Pembersihan 50:50 Selesai!")
print(f"Jumlah baris awal: {jumlah_awal}")
print(f"Baris dihapus: {jumlah_awal - len(data)}")
print(f"Sisa data penelitian: {len(data)} baris")

#DATA NORMALIZATION
# penyeragaman teks
# case folding
# emoji 

#FEATURE ENGINEERING

Pembersihan 50:50 Selesai!
Jumlah baris awal: 25564
Baris dihapus: 3285
Sisa data penelitian: 22279 baris


In [5]:
#NORMALISASI DATA
import re
#PENYERAGAMAN TEKS
# 1. Pastikan semua data di kolom text adalah string dan tangani nilai kosong (NaN)
data['text'] = data['text'].astype(str).fillna('')

# 2. Jalankan kembali fungsi normalisasi yang sudah diperbaiki
def data_normalization(text):
    # Tambahkan pengecekan tipe data untuk keamanan ekstra
    if not isinstance(text, str):
        text = str(text)
        
    # Case Folding (Bab 3.2.2)
    text = text.lower()
    
    # Ganti URL
    text = re.sub(r'http\S+|www\S+|https\S+', 'url', text, flags=re.MULTILINE)
    
    # Ganti Mention (@) menjadi username
    text = re.sub(r'@\w+', 'username', text)
    
    # Ganti Hashtag (#)
    text = re.sub(r'#\w+', 'hashtag', text)
    
    # Menghapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Terapkan ke dataset
data['text'] = data['text'].apply(data_normalization)
print(data['text'])
print("Normalisasi Berhasil!")

0        wajah jurnalis dibalut perban saat melaporkan ...
1        elektabilitas paslon 02 sebentar lagi terjun b...
4        kristen, hindu, islam dapat perlakuan istimewa...
8        cina mengirimkan satelit penginderaan jauh bar...
12       bisa bisanya aing masuk ke ruangan bos ngucapi...
                               ...                        
28442      username abis packing nnt ingfokan cowo kristen
28443            akun whatsapp pj bupati tegal agustyarsah
28444    buldoser israel menabrak pengungsi pasien ruma...
28446    perayaan chinese new year tahun ini ya akan ma...
28447    knrp hashtag breaking news jumlah korban jiwa ...
Name: text, Length: 22279, dtype: object
Normalisasi Berhasil!


In [6]:
#feature engineering

import pandas as pd
import numpy as np

def run_feature_engineering(df):
    # --- 1. Fungsi untuk Soft Label (Label Distribution Learning) ---
    def to_soft_label(labels):
        if isinstance(labels, list) and len(labels) > 0:
            total_anotator = len(labels)
            # Menghitung jumlah anotator yang memberikan nilai 1
            count_1 = sum(1 for i in labels if str(i) == '1')
            
            # Rasio untuk kelas positif (1) dan negatif (0)
            ratio_1 = round(count_1 / total_anotator, 2)
            ratio_0 = round(1.0 - ratio_1, 2)
            
            # Format output berupa distribusi probabilitas [prob_1, prob_0]
            # Contoh kasus: [1, 0, 0, 1] -> count_1 = 2, total = 4 -> [0.5, 0.5] (50:50)
            return [ratio_1, ratio_0]
        return [0.0, 1.0] # Default jika data kosong

    # --- 2. Fungsi untuk Hard Label (Majority Voting / Faktor Agregat) ---
    def to_hard_label(labels):
        if isinstance(labels, list) and len(labels) > 0:
            total_anotator = len(labels)
            count_1 = sum(1 for i in labels if str(i) == '1')
            
            # Majority Voting: Jika rasio nilai 1 lebih dari 50%, maka label final adalah 1
            # Jika seimbang atau lebih banyak 0, maka label final adalah 0
            return 1 if (count_1 / total_anotator) > 0.5 else 0
        return 0

    # --- 3. Eksekusi Pembuatan Kolom Baru ---
    # Membuat kolom Soft Label
    df['soft_toxic'] = df['toxicity'].apply(to_soft_label)
    df['soft_polarized'] = df['polarized'].apply(to_soft_label)
    
    # Membuat kolom Hard Label
    df['hard_toxic'] = df['toxicity'].apply(to_hard_label)
    df['hard_polarized'] = df['polarized'].apply(to_hard_label)
    
    return df

# Terapkan fungsi ke dataset kamu
data = run_feature_engineering(data)

# --- 4. Cek Hasil Perubahan ---
print("--- Hasil Feature Engineering ---")
print(data[['text', 'toxicity', 'soft_toxic', 'hard_toxic']].head())

--- Hasil Feature Engineering ---
                                                 text toxicity  soft_toxic  \
0   wajah jurnalis dibalut perban saat melaporkan ...   [0, 0]  [0.0, 1.0]   
1   elektabilitas paslon 02 sebentar lagi terjun b...   [0, 0]  [0.0, 1.0]   
4   kristen, hindu, islam dapat perlakuan istimewa...   [0, 0]  [0.0, 1.0]   
8   cina mengirimkan satelit penginderaan jauh bar...   [0, 0]  [0.0, 1.0]   
12  bisa bisanya aing masuk ke ruangan bos ngucapi...   [0, 0]  [0.0, 1.0]   

    hard_toxic  
0            0  
1            0  
4            0  
8            0  
12           0  


In [7]:
print(data)

                                                    text toxicity polarized  \
0      wajah jurnalis dibalut perban saat melaporkan ...   [0, 0]    [0, 0]   
1      elektabilitas paslon 02 sebentar lagi terjun b...   [0, 0]    [1, 1]   
4      kristen, hindu, islam dapat perlakuan istimewa...   [0, 0]    [1, 1]   
8      cina mengirimkan satelit penginderaan jauh bar...   [0, 0]    [0, 0]   
12     bisa bisanya aing masuk ke ruangan bos ngucapi...   [0, 0]    [0, 0]   
...                                                  ...      ...       ...   
28442    username abis packing nnt ingfokan cowo kristen      [0]       [0]   
28443          akun whatsapp pj bupati tegal agustyarsah      [0]       [0]   
28444  buldoser israel menabrak pengungsi pasien ruma...      [0]       [1]   
28446  perayaan chinese new year tahun ini ya akan ma...      [0]       [0]   
28447  knrp hashtag breaking news jumlah korban jiwa ...      [0]       [0]   

      annotators_id is_noise_or_spam_text  soft_tox

In [8]:
#data splitting 80:10:10.  70:15:15 , 90:5:5
from sklearn.model_selection import train_test_split

train_data1, temp_data1 = train_test_split(data, test_size=0.3, random_state=42)
val_data1, test_data1 = train_test_split(temp_data1, test_size=0.5, random_state=42)
print(f"Jumlah data training 70:15:15 : {len(train_data1)}")
print(f"Jumlah data validation 70:15:15 : {len(val_data1)}")
print(f"Jumlah data testing 70:15:15 : {len(test_data1)}")

print("\n")

train_data2, temp_data2 = train_test_split(data, test_size=0.2, random_state=42)
val_data2, test_data2 = train_test_split(temp_data2, test_size=0.5, random_state=42)
print(f"Jumlah data training 80:10:10 : {len(train_data2)}")
print(f"Jumlah data validation 80:10:10 : {len(val_data2)}")
print(f"Jumlah data testing 80:10:10 : {len(test_data2)}")

print("\n")

train_data3, temp_data3 = train_test_split(data, test_size=0.1, random_state=42)
val_data3, test_data3 = train_test_split(temp_data3, test_size=0.5, random_state=42)
print(f"Jumlah data training 90:5:5 : {len(train_data3)}")
print(f"Jumlah data validation 90:5:5 : {len(val_data3)}")
print(f"Jumlah data testing 90:5:5 : {len(test_data3)}")



Jumlah data training 70:15:15 : 15595
Jumlah data validation 70:15:15 : 3342
Jumlah data testing 70:15:15 : 3342


Jumlah data training 80:10:10 : 17823
Jumlah data validation 80:10:10 : 2228
Jumlah data testing 80:10:10 : 2228


Jumlah data training 90:5:5 : 20051
Jumlah data validation 90:5:5 : 1114
Jumlah data testing 90:5:5 : 1114


In [9]:
print("Distribusi train_data1:\n", train_data1['hard_toxic'].value_counts())
print("\nDistribusi train_data2:\n", train_data2['hard_toxic'].value_counts())
print("\nDistribusi train_data3:\n", train_data3['hard_toxic'].value_counts())

Distribusi train_data1:
 hard_toxic
0    14318
1     1277
Name: count, dtype: int64

Distribusi train_data2:
 hard_toxic
0    16392
1     1431
Name: count, dtype: int64

Distribusi train_data3:
 hard_toxic
0    18460
1     1591
Name: count, dtype: int64


In [10]:
#RUS rasio 1:3
from imblearn.under_sampling import RandomUnderSampler

# Strategi 0.333 sudah tepat untuk rasio 1:3
rus = RandomUnderSampler(sampling_strategy=0.333, random_state=42)

# PERBAIKAN: Tambahkan y_train_rus1
X_train_rus_toxic1, y_train_rus_toxic1 = rus.fit_resample(train_data1['text'].values.reshape(-1, 1), train_data1['hard_toxic'])
print(f"Jumlah data training setelah RUS 1: {len(X_train_rus_toxic1)}")
    
X_train_rus_toxic2, y_train_rus_toxic2 = rus.fit_resample(train_data2['text'].values.reshape(-1, 1), train_data2['hard_toxic'])
print(f"Jumlah data training setelah RUS 2: {len(X_train_rus_toxic2)}")

X_train_rus_toxic3, y_train_rus_toxic3 = rus.fit_resample(train_data3['text'].values.reshape(-1, 1), train_data3['hard_toxic'])
print(f"Jumlah data training setelah RUS 3: {len(X_train_rus_toxic3)}")

X_train_rus_polarized1, y_train_rus_polarized1 = rus.fit_resample(train_data1['text'].values.reshape(-1, 1), train_data1['hard_polarized'])
print(f"Jumlah data training polarized setelah RUS 1: {len(X_train_rus_polarized1)}")

X_train_rus_polarized2, y_train_rus_polarized2 = rus.fit_resample(train_data2['text'].values.reshape(-1, 1), train_data2['hard_polarized'])
print(f"Jumlah data training polarized setelah RUS 2: {len(X_train_rus_polarized2)}")

X_train_rus_polarized3, y_train_rus_polarized3 = rus.fit_resample(train_data3['text'].values.reshape(-1, 1), train_data3['hard_polarized'])
print(f"Jumlah data training polarized setelah RUS 3: {len(X_train_rus_polarized3)}")

Jumlah data training setelah RUS 1: 5111
Jumlah data training setelah RUS 2: 5728
Jumlah data training setelah RUS 3: 6368
Jumlah data training polarized setelah RUS 1: 9603
Jumlah data training polarized setelah RUS 2: 11020
Jumlah data training polarized setelah RUS 3: 12409


In [11]:
# Distribusi setelah RUS
print("\nDistribusi setelah RUS 1 (Toxicity):\n", pd.Series(y_train_rus_toxic1).value_counts())
print("\nDistribusi setelah RUS 2 (Toxicity):\n", pd.Series(y_train_rus_toxic2).value_counts())
print("\nDistribusi setelah RUS 3 (Toxicity):\n", pd.Series(y_train_rus_toxic3).value_counts())
print("\nDistribusi setelah RUS 1 (Polarized):\n", pd.Series(y_train_rus_polarized1).value_counts())
print("\nDistribusi setelah RUS 2 (Polarized):\n", pd.Series(y_train_rus_polarized2).value_counts())
print("\nDistribusi setelah RUS 3 (Polarized):\n", pd.Series(y_train_rus_polarized3).value_counts())


Distribusi setelah RUS 1 (Toxicity):
 hard_toxic
0    3834
1    1277
Name: count, dtype: int64

Distribusi setelah RUS 2 (Toxicity):
 hard_toxic
0    4297
1    1431
Name: count, dtype: int64

Distribusi setelah RUS 3 (Toxicity):
 hard_toxic
0    4777
1    1591
Name: count, dtype: int64

Distribusi setelah RUS 1 (Polarized):
 hard_polarized
0    7204
1    2399
Name: count, dtype: int64

Distribusi setelah RUS 2 (Polarized):
 hard_polarized
0    8267
1    2753
Name: count, dtype: int64

Distribusi setelah RUS 3 (Polarized):
 hard_polarized
0    9309
1    3100
Name: count, dtype: int64


In [ ]:
# implementasi
print(data)

                                                    text toxicity polarized  \
0      wajah jurnalis dibalut perban saat melaporkan ...   [0, 0]    [0, 0]   
1      elektabilitas paslon 02 sebentar lagi terjun b...   [0, 0]    [1, 1]   
4      kristen, hindu, islam dapat perlakuan istimewa...   [0, 0]    [1, 1]   
8      cina mengirimkan satelit penginderaan jauh bar...   [0, 0]    [0, 0]   
12     bisa bisanya aing masuk ke ruangan bos ngucapi...   [0, 0]    [0, 0]   
...                                                  ...      ...       ...   
28442    username abis packing nnt ingfokan cowo kristen      [0]       [0]   
28443          akun whatsapp pj bupati tegal agustyarsah      [0]       [0]   
28444  buldoser israel menabrak pengungsi pasien ruma...      [0]       [1]   
28446  perayaan chinese new year tahun ini ya akan ma...      [0]       [0]   
28447  knrp hashtag breaking news jumlah korban jiwa ...      [0]       [0]   

      annotators_id is_noise_or_spam_text  soft_tox